In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from ModuleFourMilestone import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "password"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.retrieve({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
print(len(df.to_dict(orient='records')))
print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode("utf-8")

# creates the filtered list for each rescue filter

# Rescue breed filters
WATER_RESCUE_BREEDS = [
    "Labrador Retriever Mix",
    "Chesapeake Bay Retriever",
    "Newfoundland"
]

MOUNTAIN_RESCUE_BREEDS = [
    "German Shepherd",
    "Alaskan Malamute",
    "Old English Sheepdog",
    "Siberian Husky",
    "Rottweiler"
]

DISASTER_TRACKING_BREEDS = [
    "German Shepherd",
    "Doberman Pinscher",
    "Golden Retriever",
    "Bloodhound",
    "Rottweiler"
]


# Animal filter function for sorting
def filter_animals(filter_type):
    if filter_type is None or filter_type == "reset":
        return df.copy()
    
    if filter_type == "water":
        breeds = WATER_RESCUE_BREEDS
        sex = "Intact Female"
        min_age = 26
        max_age = 156
        
    elif filter_type == "mountain":
        breeds = MOUNTAIN_RESCUE_BREEDS
        sex = "Intact Male"
        min_age = 26
        max_age = 156
        
    elif filter_type == "disaster":
        breeds = DISASTER_TRACKING_BREEDS
        sex = "Intact Male"
        min_age = 20
        max_age = 300
        
    else:
        return df.copy()
    
    # normalize breed names for comparison
    normalized_breeds = [
        breed.lower() for breed in breeds
    ]
    
    # convert ages to a numeric value
    ages = pd.to_numeric(
        df["age_upon_outcome_in_weeks"],
        errors="coerce"
    )
    
    # filters queries based on required matching data
    filter_df = df[
        (df["animal_type"].astype(str).str.lower() == "dog")
        &
        (
            df["breed"]
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(normalized_breeds)
        )
        &
        (
            df["sex_upon_outcome"].astype(str).str.strip().str.lower() == sex.lower()
        )
        &
        ages.between(min_age, max_age)
    ]
    
        
    return filter_df.copy()

    


app.layout = html.Div([
    html.Div(
        html.A(
            html.Img(
                src=(
                    "data:image/png;base64,"
                    + encoded_image
                ),
                style={
                    "width": "200px",
                    "height": "auto",
                    "display": "block",
                    "margin": "0 auto"
                }
            ),
            href="https://www.snhu.edu",
            target="_blank"
        ),
        style={"textAlign": "center"}
    ),
    
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('SNHU CS-340'))),
    html.Center(html.B(html.H2('Dashboard V1 - Kyle Davis'))),
    html.Hr(),
    # creates the category box for filtering for different rescue breeds
    html.Div(
        [
            html.Label("Select a Rescue Category:", style={"fontWeight": "bold"}),
            dcc.RadioItems(id="filter-type", 
                           options=[{"label": "Water Rescue", "value": "water"},
                                    {"label": "Mountain / Wilderness Rescue", "value": "mountain"},
                                    {"label": "Disaster / Tracking Rescue", "value": "disaster"},
                                    {"label": "Reset - Show all", "value": "reset"}],
                           value="reset",
                           inline=False,
                           
                           labelStyle={
                               "display": "block",
                               "margin": "8px 0"
                           }
                          )
        ],
        style={"padding": "10px"}
    ),
    html.Hr(),
    
    # sets features for data table along with styling
    #Set up the features for your interactive data table to make it user-friendly for your client
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        filter_action="native",
        sort_action="native",
        column_selectable=False,
        row_selectable="single",
        selected_columns=[],
        selected_rows=[0],
        page_action="native",
        page_current=0,
        page_size=10,
        
        style_table={
            'overflowX': 'auto'
        },
        
        style_cell={
            'textAlign': 'left',
            'minWidth': '100px',
            'width': '100px',
            'maxWidth': '150px',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis',
            
        },
        
        style_header={
            'fontWeight': 'bold'
        }

                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex', "flexWrap": "wrap", "gap": "20px", "alignItems": "flex-start"},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',
            style={
                "flex": "1 1 450px",
                "minWidth": "0"
            }

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            style={
                "flex": "1 1 450px",
                "minWidth": "0"
            }
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    filter_df = filter_animals(filter_type)
    
    return filter_df.to_dict("records")

@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if not viewData:
        return [
            dcc.Graph(
                figure=px.bar(
                    title="Animals by Breed - No Data Available"
                )
            )
        ]
    
    chart_df = pd.DataFrame.from_records(viewData)
    
    breed_counts = (
        chart_df["breed"]
        .fillna("Unknown")
        .value_counts()
        .head(15)
        .rename_axis("Breed")
        .reset_index(name="Count")
    )
    
    figure = px.bar(
        breed_counts,
        x="Breed",
        y="Count",
        title="Top 15 Breeds in Current Results"
    )
    
    return [dcc.Graph(figure=figure, style={"height": "500", "width": "100%"})]

@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '100%', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

10000
Index(['rec_num', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed',
       'color', 'date_of_birth', 'datetime', 'monthyear', 'name',
       'outcome_subtype', 'outcome_type', 'sex_upon_outcome', 'location_lat',
       'location_long', 'age_upon_outcome_in_weeks'],
      dtype='object')
Dash app running on https://jordaneffect-fluidfax-3000.codio.io/proxy/8050/
